# Association Rule Mining: Patterns Predicting Diabetes

**Input:** `data/processed/CDC_Diabetes_Dataset_clean.csv`

**Purpose:** Use the Apriori algorithm (mlxtend) to discover frequent itemset combinations in binary health and lifestyle indicators that are associated with a diabetes or pre-diabetes outcome. Rules are filtered to those with `Diabetes=1` as the consequent and ranked by lift and confidence.

**Output:** Top rules saved to `data/processed/ARM_Top_Rules.csv` for use in the project report.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "CDC_Diabetes_Dataset_clean.csv"

FIG_DIR = PROJECT_ROOT / "figures" / "results_ARM"
FIG_DIR.mkdir(parents=True, exist_ok=True)


print("Project root directory:", PROJECT_ROOT)
print("Data path exists:", DATA_PATH.exists())
assert DATA_PATH.exists(), f"Data file not found at {DATA_PATH}"
print("Figures directory:", FIG_DIR)

In [ ]:
import pandas as pd
import numpy as np

from mlxtend.frequent_patterns import apriori, association_rules


pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 200)

In [ ]:
# Load data
# load data from csv
df = pd.read_csv(DATA_PATH)
print("Data loaded successfully.")
print("Top 5 rows:")
df.head()

In [ ]:
# target definition + overview 

# Define binary diabetes target for ARM
df = df.copy()
df["Diabetes"] = (df["Diabetes_012"] > 0).astype(int)

# Display class distribution
target_summary = (
    df["Diabetes"]
    .value_counts(normalize=True)
    .rename("proportion")
    .to_frame()
)

display(target_summary)

print("Target definition:")
print("0 = No diabetes")
print("1 = Pre-diabetes or diabetes")

In [ ]:

# Target 
target = "Diabetes"

# Binary health indicators (0/1)
binary_features = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost",
    "DiffWalk", "Sex"
]

# Ordinal / categorical features (coded levels)
ordinal_features = [
    "GenHlth",    # 1 (Excellent) – 5 (Poor)
    "Age",        # age brackets (1–13)
    "Education",  # education levels
    "Income"      # income brackets
]

# Continuous features
continuous_features = [
    "BMI",
    "MentHlth",
    "PhysHlth"
]

# Sanity check: confirm all features exist
all_features = binary_features + ordinal_features + continuous_features + [target]
missing = [c for c in all_features if c not in df.columns]

print("Missing columns:", missing)
print("\nBinary features:", binary_features)
print("\nOrdinal features:", ordinal_features)
print("\nContinuous features:", continuous_features)

In [ ]:
# create binned / labelled versions for ARM itemisation

arm = df.copy()

# --- BMI bins (clinically meaningful) ---
bmi_bins = [-np.inf, 18.5, 25, 30, 35, 40, np.inf]
bmi_labels = ["Underweight", "Normal", "Overweight", "Obese_I", "Obese_II", "Obese_III"]
arm["BMI_bin"] = pd.cut(arm["BMI"], bins=bmi_bins, labels=bmi_labels)

# --- Mental health days bins (0-30 days of poor mental health) ---
mh_bins = [-np.inf, 0, 5, 15, 30]
mh_labels = ["0", "1-5", "6-15", "16-30"]
arm["MentHlth_bin"] = pd.cut(arm["MentHlth"], bins=mh_bins, labels=mh_labels)

# --- Physical health days bins ---
ph_bins = [-np.inf, 0, 5, 15, 30]
ph_labels = ["0", "1-5", "6-15", "16-30"]
arm["PhysHlth_bin"] = pd.cut(arm["PhysHlth"], bins=ph_bins, labels=ph_labels)

# Keep ordinal vars as categorical (labels optional; codes are still interpretable as brackets/levels)
for c in ordinal_features:
    arm[c] = arm[c].astype("category")

# Quick check: show distributions of the binned vars
display(arm["BMI_bin"].value_counts(dropna=False))
display(arm["MentHlth_bin"].value_counts(dropna=False))
display(arm["PhysHlth_bin"].value_counts(dropna=False))

arm[["BMI", "BMI_bin", "MentHlth", "MentHlth_bin", "PhysHlth", "PhysHlth_bin"]].head(10)

In [ ]:
# build basket (boolean item matrix)

# binary items: keep only the positive state as an item (e.g., HighBP=1)
basket_bin = pd.DataFrame(index=arm.index)
for c in binary_features:
    basket_bin[f"{c}=1"] = (arm[c] == 1)

# categorical items: one-hot encode ordinal + binned continuous
cat_cols = ordinal_features + ["BMI_bin", "MentHlth_bin", "PhysHlth_bin"]
basket_cat = pd.get_dummies(arm[cat_cols], prefix=cat_cols, prefix_sep="=")

# target item (RHS): Diabetes=1
basket_target = pd.DataFrame({f"{target}=1": (arm[target] == 1)}, index=arm.index)

# Combine into final basket
basket = pd.concat([basket_bin, basket_cat, basket_target], axis=1).astype(bool)

print("Basket shape:", basket.shape)
print("Example item columns:", basket.columns[:15].tolist())
basket.head()

In [ ]:
#  stratified sampling + Apriori frequent itemsets


rhs_item = f"{target}=1"

# --- Stratified sample indices (keeps Diabetes proportion) ---
n_sample = 50000  # low n as my whole system keeps crashing
pos_idx = basket.index[basket[rhs_item]].to_numpy()
neg_idx = basket.index[~basket[rhs_item]].to_numpy()

# keep original class balance
p = len(pos_idx) / len(basket)
n_pos = int(n_sample * p)
n_neg = n_sample - n_pos

rng = np.random.default_rng(42)
sample_idx = np.concatenate([
    rng.choice(pos_idx, size=n_pos, replace=False),
    rng.choice(neg_idx, size=n_neg, replace=False),
])

basket_s = basket.loc[sample_idx].copy()

print("Sample basket shape:", basket_s.shape)
print(f"Sample P({rhs_item}) =", basket_s[rhs_item].mean().round(4))

# --- Frequent itemsets on sample ---
min_support = 0.02
max_len = 3

freq_itemsets = apriori(
    basket_s,
    min_support=min_support,
    use_colnames=True,
    max_len=max_len
).sort_values("support", ascending=False)

print("Frequent itemsets:", freq_itemsets.shape)
freq_itemsets.head(10)

In [ ]:
# association rules (filter to consequent Diabetes=1)


rhs_item = f"{target}=1"

rules = association_rules(freq_itemsets, metric="confidence", min_threshold=0.2)

# Keep only rules where consequent is exactly {Diabetes=1}
rules_diab = rules[rules["consequents"].apply(lambda s: (len(s) == 1) and (rhs_item in s))].copy()

# Add lengths for filtering
rules_diab["antecedent_len"] = rules_diab["antecedents"].apply(len)

# Filter to reduce noise / triviality
rules_diab_f = rules_diab[
    (rules_diab["support"] >= 0.01) &
    (rules_diab["lift"] >= 1.2) &
    (rules_diab["antecedent_len"] <= 3)
].sort_values(["lift", "confidence", "support"], ascending=False)

print("All diabetes rules:", rules_diab.shape)
print("Filtered diabetes rules:", rules_diab_f.shape)

rules_diab_f.head(15)[["antecedents", "consequents", "support", "confidence", "lift"]]

In [ ]:
# format top rules for report ready

def pretty_itemset(s):
    return ", ".join(sorted(list(s)))

report_rules = rules_diab_f.copy()
report_rules["Rule (Antecedent → Diabetes)"] = report_rules["antecedents"].apply(pretty_itemset) + "  →  " + rhs_item

# Select + round for readability
top_rules = report_rules[
    ["Rule (Antecedent → Diabetes)", "support", "confidence", "lift", "antecedent_len"]
].head(12).copy()

top_rules["support"] = top_rules["support"].round(4)
top_rules["confidence"] = top_rules["confidence"].round(3)
top_rules["lift"] = top_rules["lift"].round(2)

display(top_rules)

In [ ]:
# save top rules table (appendix-ready)

out_path = PROJECT_ROOT / "data" / "processed" / "ARM_Top_Rules.csv"
top_rules.to_csv(out_path, index=False)
print("Saved:", out_path)

In [ ]:
# Cell 10: add counts for interpretability

N = len(basket_s)
top_rules2 = top_rules.copy()
top_rules2.insert(1, "count_in_sample", (top_rules2["support"] * N).round().astype(int))

display(top_rules2)